This script appends the h2-distribution matrix to the IAM output file

In [ ]:
import pandas as pd
from pathlib import Path

In [35]:
# Load matrix and IAM output file
df_h2 = pd.read_excel(
    r"C:\Users\ac145674\Downloads\h2_distribution_matrix_IAM_by-year.xlsx",
    sheet_name="h2-distribution",
 )
#
# df_iam = pd.read_csv(
#    r"C:\Users\ac145674\Downloads\message_SSP2-M.csv"
# )



In [ ]:
# -- Import any file in the iam_output_files folder, to which the distribution matrix will be appended to. --
def load_iam_file(filepath):
    ext = Path(filepath).suffix.lower()
    if ext == ".csv":
        return pd.read_csv(filepath)
    elif ext == ".mif":
        return pd.read_csv(filepath, sep=";")
    elif ext == ".xlsx":
        return pd.read_excel(filepath)
    else:
        raise ValueError(f"Unsupported file format: {ext}")


supported = [".csv", ".mif", ".xlsx"]
input_folder = Path(r"C:\Users\ac145674\coding_projects\premise_h2-distribution\premise\data\iam_output_files")

iam_file = next(
    (f for f in input_folder.iterdir() if f.suffix.lower() in supported), None
)

if iam_file is None:
    raise FileNotFoundError("No supported IAM file found in input folder")

df_iam = load_iam_file(iam_file)
print(f"Loaded: {iam_file.name}")



Loaded: REMIND_generic_SSP2-NPi2025.mif


In [37]:
# ── Fill Scenario and Model from iam output file to h2 distribution matrix ──────────────────────────────────────────────
# Model and Scenario are constant across all iam rows
# quick sanity check, so there is only one scenario per iam-output-file.
assert (
    df_iam["Scenario"].nunique() == 1
), f"Expected 1 scenario, found {df_csv['Scenario'].nunique()}"
model_value = df_iam["Model"].dropna().iloc[0] 
scenario_value = (df_iam["Scenario"].dropna().iloc[0])  # could also use .unique()[0]

In [38]:
# -- some data analysis --
regions = sorted(df_iam["Region"].dropna().unique())
scenario = sorted(df_iam["Scenario"].dropna().unique())
model = sorted(df_iam["Model"].dropna().unique())
variables = sorted(df_iam["Variable"].dropna().unique())
#regions
# scenario
# model
# variables

In [39]:
# Make sure new variables from iam output matrix also have Model, Scenario and Region columns
df_h2 = df_h2.loc[df_h2.index.repeat(len(regions))].reset_index(drop=True)
df_h2["Region"] = regions * (len(df_h2) // len(regions))
df_h2["Model"] = model_value
df_h2["Scenario"] = scenario_value

In [40]:
# --- Drop index and 0 columns in the iam output file ---
# Rename integer year columns to strings so they align with the CSV
df_h2.columns = [str(c) for c in df_h2.columns]
# Drop the unnamed index column
if "Unnamed: 0" in df_iam.columns:
    df_iam = df_iam.drop(columns=["Unnamed: 0"])

In [41]:
# --- Cleaning columns ---
# Align column order
meta_cols = ["Model", "Scenario", "Region", "Variable", "Unit"]

# Union of year columns sorted numerically
year_cols_h2 = sorted([c for c in df_h2.columns if c.isdigit()], key=int)
year_cols_iam = sorted([c for c in df_iam.columns if c.isdigit()], key=int)
all_year_cols = sorted(set(year_cols_h2) | set(year_cols_iam), key=int)

# Reindex to full column set (NaN -> 0.0)
df_h2 = df_h2.reindex(columns=meta_cols + all_year_cols)
df_iam = df_iam.reindex(columns=meta_cols + all_year_cols)

In [42]:
# --- Merging dfs ---
df_merge = pd.concat([df_iam, df_h2], ignore_index=True)
# remove NaN in year columns
df_merge[all_year_cols] = df_merge[all_year_cols].fillna(0.0).astype(float)
df_merge

,Model,Scenario,Region,Variable,Unit,2005,2010,2015,2020,2025,...,2050,2055,2060,2070,2080,2090,2100,2110,2130,2150
0,REMIND,SSP2-NPi2025,CAZ,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,REMIND,SSP2-NPi2025,CHA,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,REMIND,SSP2-NPi2025,EUR,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,REMIND,SSP2-NPi2025,IND,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,REMIND,SSP2-NPi2025,JPN,CDR|OAE quicklime,Mt CaO/yr,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81901,REMIND,SSP2-NPi2025,OAS,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81902,REMIND,SSP2-NPi2025,REF,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81903,REMIND,SSP2-NPi2025,SSA,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81904,REMIND,SSP2-NPi2025,USA,Hydrogen Distribution | NH3 | Other | Ship,%,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
# --- Export merged csv ---
# define output-path
output_path = iam_file
df_merge.to_csv(output_path, index=False)

# if input file was a .mif or .xlsx file:
ext = iam_file.suffix.lower()
if ext in [".csv", ".mif"]:
    sep = ";" if ext == ".mif" else ","
    df_merge.to_csv(iam_file, index=False, sep=sep)
elif ext == ".xlsx":
    df_merge.to_excel(iam_file, index=False)

# additional info
print(f"Table merged and save to {output_path}")
print(
    f"Shape: {df_merge.shape}  (expected rows added: {len(df_h2)} = {len(df_h2)//len(regions)} variables × {len(regions)} regions)"
)
print(f"\nModel:    {model_value}")
print(f"Scenario: {scenario_value}")
print(f"Regions:  {regions}")

Table merged and save to C:\Users\ac145674\coding_projects\premise_h2-distribution\premise\data\iam_output_files\REMIND_generic_SSP2-NPi2025.mif
Shape: (81906, 24)  (expected rows added: 585 = 45 variables × 13 regions)

Model:    REMIND
Scenario: SSP2-NPi2025
Regions:  ['CAZ', 'CHA', 'EUR', 'IND', 'JPN', 'LAM', 'MEA', 'NEU', 'OAS', 'REF', 'SSA', 'USA', 'World']
